In [ ]:
import numpy as np
import pandas as pd
import os
ifiles = os.listdir('idle')
idle_csv = pd.read_csv('idle/idle_2200.csv')
NUMCORE=28

In [ ]:
idle_csv
util = idle_csv['cycle-count:10'] / (idle_csv['Freq(kHz):10'] * idle_csv['Duration(ms)'])
upi = idle_csv['uops_executed.core:11'] / idle_csv['instructions:11']
util.plot()

In [ ]:
def merge_csvs(path):
    files = os.listdir(path)
    df = pd.DataFrame()
    files.sort()
    print(files)
    for f in files:
        if f.endswith('.csv'):
            temp = pd.read_csv(path + '/' + f)
            df = pd.concat([df, temp])
    df.reset_index(drop=True, inplace=True)
    return df
idle_csv = merge_csvs('idle')

In [ ]:
def filter_idle(df):
    df['avgutil'] = pd.Series(0.0, index=df.index)
    for core in range(NUMCORE):
        df['avgutil'] += idle_csv['cycle-count:'+str(core)] / (idle_csv['Freq(kHz):'+str(core)])
    df['avgutil'] = df['avgutil']/(NUMCORE * df['Duration(ms)'])
    #df['avgutil'].plot()
    df = df[df['Package Power(W):0'] > 0]
    return df[df['avgutil'] < 0.01]
idle_df = filter_idle(idle_csv)
idle_df['Package Power(W):0'].plot()
#idle_df['Temp(C):0'].plot()


In [ ]:
from PPEPDataset import PPEPRecord

idle_record = PPEPRecord(idle_df)

In [ ]:
pd.Series(idle_record.coredata[:,0,0,0]).plot()

In [ ]:

idle_csv.plot.scatter('Freq(kHz):0', 'Voltage:0')

In [ ]:
idle_df['Vavg:0'] = 0.0

for i in range(0,NUMCORE):
    idle_df['Vavg:0'] = idle_df['Vavg:0'] + idle_df['Voltage:'+str(i)] / NUMCORE



idle_df.plot.scatter('Vavg:0', 'Package Power(W):0')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.pipeline import make_pipeline

x = idle_df['Vavg:0'].to_numpy()
y = idle_df['Package Power(W):0'].to_numpy()


X_train = x[:, np.newaxis]
# Transform features to polynomial
model = make_pipeline(PolynomialFeatures(3), Ridge(alpha=100,fit_intercept=False,positive=True))
model.fit(X_train, y)
print(model.get_params())
print(model.get_params()['ridge'].coef_)
print(model.get_params()['ridge'].intercept_)

mymodel = np.poly1d(model.get_params()['ridge'].coef_[::-1])
myline = np.linspace(0.60, 0.85, 100)
y_plot = model.predict(myline[:, np.newaxis])


plt.scatter(x, y, color='blue')
plt.plot(myline, mymodel(myline))
#plt.plot(myline, y_plot,color='red')
#plt.plot(x, X_poly)
plt.show()


with open('idlemodel.txt', 'w') as imodel:
    for c in mymodel.coef:
        imodel.write(str(c) + '\n')


In [ ]:
tempdiff = (mymodel(x) - y)

idle_df['Tavg:0'] = 0.0

for i in range(0,NUMCORE):
    idle_df['Tavg:0'] = idle_df['Tavg:0'] + idle_df['Temp(C):'+str(i)] / NUMCORE

ptmp = idle_df['Tavg:0'].to_numpy() + 273

plt.scatter(ptmp,tempdiff)



In [ ]:
turbo_csv = pd.read_csv('active/densenet121_bf16_turbo.csv')

freq0 = idle_csv['Freq(kHz):0'].to_numpy()*1e-6
volt0 = idle_csv['Voltage:0'].to_numpy()

F_train = freq0[:, np.newaxis]
# Transform features to polynomial
model = make_pipeline(PolynomialFeatures(2), Ridge(alpha=0.1,fit_intercept=False, positive=True))
model.fit(F_train, volt0)
print(model.get_params())
print(model.get_params()['ridge'].coef_)

poly1d_fn = np.poly1d(model.get_params()['ridge'].coef_[::-1])
myline = np.linspace(1, 3.5, 100)

print(poly1d_fn)
print(myline)

freqt = turbo_csv['Freq(kHz):0'].to_numpy()
voltt = turbo_csv['Voltage:0'].to_numpy()

plt.scatter(freq0, volt0, color='blue')
#plt.scatter(freq0, poly1d_fn(freq0), color='red')
plt.plot(myline, poly1d_fn(myline), color='red')
plt.show()
with open('vfpoly.txt', 'w') as vfpoly:
    for c in poly1d_fn.coef:
        vfpoly.write(str(c) + '\n')



In [ ]:
gnn_tts = pd.read_csv('active/densenet121_bf16_turbo.csv')
gtrecord = PPEPRecord(gnn_tts)
print(gtrecord.coredata.shape)
print(gtrecord.corestat.shape)
print(gtrecord.pkgdata.shape)
pkgctrdata = gtrecord.coredata.sum(axis=2)
pkgstatdata = gtrecord.corestat.mean(axis=2)
print(gtrecord.counters)
print(gtrecord.stats)
print(pkgctrdata.shape)
print(pkgstatdata.shape)
pkgstatdata[:,:,3] *= 10 # bips is sum
pkgctrdata[:,:,0] *= 0.1 # voltage is average
pkgctrdata[:,:,1] *= 1e-7 # Freq is average and convert to ghz
pkgctrdata[:,:,2] *= 0.1 # Temp is average

volt = pd.Series(pkgctrdata[:,0, 0])
freq = pd.Series(pkgctrdata[:,0, 1])
st = pd.Series(pkgstatdata[:,0, 2])
#volt.plot()
#freq.plot()
#st.plot()

In [ ]:
pkgpower = gtrecord.pkgdata[:,:,1]
#pd.Series(pkgpower[:,1]).plot()
pkgvolt = pkgctrdata[:,:, 0]
pkgbips = pkgstatdata[:,:,3]
pkgutil = pkgstatdata[:,:,0]
pkgbcps = pkgutil * pkgctrdata[:,:,1] * 10 # bcps is also sum
pkgstats = pkgstatdata[:,:,4:] * pkgbips[:,:,None]
pkgstats = np.concatenate([pkgstats, pkgbips[:,:,None], pkgbcps[:,:,None]], axis=-1)

In [ ]:
pd.Series(pkgbcps[:,0]).plot()

In [ ]:
from sklearn import linear_model
from PPEPDataset import PPEPData

#mydata = PPEPData(['llama_stablediffusion_2600.csv', 'gnn_tts_2600.csv'])
trainlist = ['active/'+fname for fname in os.listdir('active') if fname.endswith('bf16_turbo.csv')]
print(trainlist)
mydata = PPEPData(trainlist)
active_model = linear_model.Ridge(alpha=1, fit_intercept=False, positive=True)
pkgvoltsquare = mydata.voltages * mydata.voltages / (0.8**2)
linearbase = (mydata.voltages + pkgvoltsquare)[:,None] * mydata.pstats
pkgidle = mydata.idlemodel(mydata.voltages)
pkgactive = mydata.power - pkgidle
active_model.fit(linearbase, pkgactive)
print(len(mydata.pstats[0]))
print(active_model.coef_)
with open('activecoef.txt', 'w') as acoef:
    for c in active_model.coef_:
        acoef.write(str(c) + '\n')


In [ ]:
'''
validdata = PPEPData(['llama_stablediffusion_2600.csv', 'gnn_tts_2600.csv'])
pkgvoltsquare = validdata.voltages * validdata.voltages
linearbase = (validdata.voltages + pkgvoltsquare)[:,None] * validdata.pstats
pkgidle = validdata.idlemodel(validdata.voltages)
pkgactive = validdata.power - pkgidle
'''


pd.Series(pkgactive[1000:2000]).plot()
apred = active_model.predict(linearbase)
pd.Series(apred[1000:2000]).plot()
aae = np.abs(pkgactive - apred)
print(aae.mean())
pkgactive.mean()


## TODO

- use multiple data, append pkg data
- validate on variable freq
- determine alpha

In [ ]:
validlist = ['active/'+fname for fname in os.listdir('active') if fname.endswith('bf16_noturbo.csv')]
print(validlist)

validdata = PPEPData(validlist)

aaes = []

for alpha in np.linspace(2,8,20):
    pkgvoltsquare = validdata.voltages ** alpha / (0.8 ** (alpha))
    linearbase = (validdata.voltages + pkgvoltsquare)[:,None] * validdata.pstats
    pkgidle = validdata.idlemodel(validdata.voltages)
    pkgactive = validdata.power - pkgidle
    apred = active_model.predict(linearbase)
    aae = np.abs(pkgactive - apred)
    aaes.append(aae.mean())
plt.plot(np.linspace(2,5,20), aaes)






In [ ]:
myalpha = 2
pkgvoltsquare = validdata.voltages ** myalpha / (0.8 ** (myalpha))
linearbase = (validdata.voltages + pkgvoltsquare)[:,None] * validdata.pstats
pkgidle = validdata.idlemodel(validdata.voltages)
pkgactive = validdata.power - pkgidle
apred = active_model.predict(linearbase)
aae = np.abs(pkgactive - apred)
pd.Series(pkgactive[5000:7000]).plot()
apred = active_model.predict(linearbase)
pd.Series(apred[5000:7000]).plot()
aae = np.abs(pkgactive - apred)
print(aae.mean())
validdata.power.mean()

In [ ]:
testlist = ['active/'+fname for fname in os.listdir('active') if fname.endswith('fp32_turbo.csv')]
print(testlist)

testdata = PPEPData(testlist)


myalpha = 2
pkgvoltsquare = testdata.voltages ** myalpha / (0.8 ** (myalpha))
linearbase = (testdata.voltages + pkgvoltsquare)[:,None] * testdata.pstats
pkgidle = testdata.idlemodel(testdata.voltages)
pkgactive = testdata.power - pkgidle
apred = active_model.predict(linearbase)
aae = np.abs(pkgactive - apred)
pd.Series(pkgactive[1000:1500]).plot()
apred = active_model.predict(linearbase)
pd.Series(apred[1000:1500]).plot()
aae = np.abs(pkgactive - apred)
print(aae.mean())
testdata.power.mean()

In [ ]:
#TODO: apply per-core active power polynomial